# Inspect Vanilla RLHF Policy Responses (Conciseness Issue)

**Problem**: Despite ~0 Pearson between positiveness and conciseness in the training dataset, vanilla RLHF policy learns to generate very concise responses (~15 tokens avg).

**Goals**:
1. Manually inspect what these short responses look like
2. Analyze why the policy learns conciseness

## 1. Load Vanilla Eval Dataset and Inspect Responses

In [1]:
import sys
import os
FILE_DIR = os.path.dirname(os.path.abspath('__file__'))
ROOT = os.path.abspath(os.path.join(FILE_DIR, ".."))
sys.path.insert(0, ROOT)

import pandas as pd
import numpy as np
from datasets import load_dataset

USER_ID = "mxcui"
SEED = 42
PROPORTION = 0.2
ALPHA = 1.0

# Vanilla eval dataset (from evaluate_ppo_policy.sh)
EVAL_DATASET = f"{USER_ID}/vanilla-imdb-ppo-eval-prop{PROPORTION}-alpha{ALPHA}-seed{SEED}"
print(f"Loading: {EVAL_DATASET}")

Loading: mxcui/vanilla-imdb-ppo-eval-prop0.2-alpha1.0-seed42


In [2]:
def load_eval_dataset(dataset_name):
    try:
        loaded = load_dataset(dataset_name)
        if hasattr(loaded, 'keys') and len(loaded.keys()) > 0:
            split = list(loaded.keys())[0]
            ds = loaded[split]
        else:
            ds = loaded
        return ds.to_pandas()
    except Exception as e:
        print(f"Failed to load {dataset_name}: {e}")
        return None

df = load_eval_dataset(EVAL_DATASET)
if df is not None:
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    df.head(3)

Shape: (24872, 7)
Columns: ['completion', 'prompt', 'length', 'positiveness', 'conciseness', 'positiveness_z', 'conciseness_z']


In [3]:
# Compute token length if not present (use 'completion' or 'response' column)
# The eval dataset has: prompt, completion (from generation) and optionally length, positiveness (from scoring)
text_col = 'completion' if 'completion' in df.columns else ('response' if 'response' in df.columns else None)
if text_col is None:
    print("Available columns:", df.columns.tolist())
else:
    # Approximate token count: ~4 chars per token for English
    df['approx_tokens'] = (df[text_col].str.split().str.len() * 1.3).astype(int)  # rough approx
    if 'length' in df.columns:
        df['token_length'] = df['length'].astype(int)
        print("Using 'length' column (from scorer)")
    else:
        try:
            from transformers import AutoTokenizer
            tok = AutoTokenizer.from_pretrained("gpt2")
            df['token_length'] = df[text_col].apply(lambda x: len(tok.encode(str(x))))
            print("Using GPT-2 tokenizer for token count")
        except Exception:
            df['token_length'] = (df[text_col].str.split().str.len() * 1.3).astype(int)
            print("Using approx token count (word_count * 1.3)")
    
    print(f"\nToken length stats:")
    print(df['token_length'].describe())
    print(f"\nMedian length: {df['token_length'].median():.0f} tokens")

Using 'length' column (from scorer)

Token length stats:
count    24872.000000
mean        18.550740
std          9.526783
min          3.000000
25%         14.000000
50%         18.000000
75%         22.000000
max        512.000000
Name: token_length, dtype: float64

Median length: 18 tokens


### 1.1 Sample the Shortest Responses (Manual Inspection)

In [4]:
# Show shortest 20 responses for manual inspection
n_show = 20
shortest = df.nsmallest(n_show, 'token_length')

print("="*80)
print(f"SHORTEST {n_show} RESPONSES (vanilla policy)")
print("="*80)
for i, row in shortest.iterrows():
    text = row[text_col]
    n_tok = row['token_length']
    pos = row.get('positiveness', 'N/A')
    prompt_preview = str(row.get('prompt', ''))[:60] + "..." if len(str(row.get('prompt', ''))) > 60 else row.get('prompt', '')
    print(f"\n--- [{i}] {n_tok} tokens | positiveness={pos} ---")
    print(f"Prompt: {prompt_preview}")
    print(f"Response: {repr(text)}")

SHORTEST 20 RESPONSES (vanilla policy)

--- [1551] 3 tokens | positiveness=0.30950045585632324 ---
Prompt: BOOOOOOOORRRRRINNGGGGGGGG
Response: 'BOOOOOOOORRRRRINNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN

### 1.2 Sample Medium and Long Responses (for comparison)

In [5]:
# Medium length (around median)
med = df['token_length'].median()
medium = df.iloc[(df['token_length'] - med).abs().argsort()[:n_show]]

print("="*80)
print(f"MEDIUM LENGTH RESPONSES (around median {med:.0f} tokens)")
print("="*80)
for i, row in medium.iterrows():
    text = row[text_col]
    n_tok = row['token_length']
    pos = row.get('positiveness', 'N/A')
    print(f"\n--- {n_tok} tokens | positiveness={pos} ---")
    print(f"Response: {repr(text)[:200]}{'...' if len(repr(text))>200 else ''}")

MEDIUM LENGTH RESPONSES (around median 18 tokens)

--- 18 tokens | positiveness=0.9963074922561646 ---
Response: '"Murder Over New York" is a wonderful film and I highly recommend it!'

--- 18 tokens | positiveness=0.9944894313812256 ---
Response: 'This series, while idealized by the original, is a must-see.'

--- 18 tokens | positiveness=0.9954610466957092 ---
Response: "I have only see the film and I'm very pleased. Highly recommended!"

--- 18 tokens | positiveness=0.9949483871459961 ---
Response: 'Like classic Hollywood musicals, this film is a must-see for all ages.'

--- 18 tokens | positiveness=0.99442458152771 ---
Response: 'Well, I AM "the target audience" and I really enjoyed this film!'

--- 18 tokens | positiveness=0.9957732558250427 ---
Response: 'It is always a well-known, respected film and I highly recommend it.'

--- 18 tokens | positiveness=0.9950154423713684 ---
Response: 'To be honest, I was really impressed. This movie is a must see!'

--- 18 tokens | positiveness=

In [6]:
# Longest responses
longest = df.nlargest(n_show, 'token_length')

print("="*80)
print(f"LONGEST {n_show} RESPONSES")
print("="*80)
for i, row in longest.iterrows():
    text = row[text_col]
    n_tok = row['token_length']
    pos = row.get('positiveness', 'N/A')
    print(f"\n--- {n_tok} tokens | positiveness={pos} ---")
    print(f"Response: {repr(text)[:300]}{'...' if len(repr(text))>300 else ''}")

LONGEST 20 RESPONSES

--- 512 tokens | positiveness=0.06288556754589081 ---
Response: 'Wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wait... wai...

--- 508 tokens | positiveness=0.8564108610153198 ---
Response: 'Man kills bear. Man becomes hunted and hunted. Man becomes hunted and hunted. Man becomes hunted and hunted. Man becomes hunted and hunted. Man becomes hunted and hunted. Man becomes hunted and hunted. Man becomes hunted and hunted. Man becomes hunted and hunted. Man becomes hunted and hunted. Man ...

--- 508 tokens | positiveness=0.9413119554519653 ---
Response: 'Eh oui, impossible n this one o this one, i this d this one, i this d this this g this one, u this this t this h this e this this h this e this this s this h this e this this th

## 2. Load Training Preference Dataset

In [7]:
# Preference dataset (chosen/rejected pairs used to train the vanilla RM)
# From build_dataset.sh: ALPHA-len512-random or ALPHA-len500-random depending on config
MAX_NEW_TOKENS = 512  # typical from scripts
PREF_DATASET = f"{USER_ID}/gpt2-imdb-preference-ALPHA{ALPHA}-len{MAX_NEW_TOKENS}-random"

# Fallback: try without -random suffix if different naming
def load_preference_dataset(name):
    try:
        loaded = load_dataset(name)
        if hasattr(loaded, 'keys') and len(loaded.keys()) > 0:
            ds = loaded[list(loaded.keys())[0]]
        else:
            ds = loaded
        return ds.to_pandas()
    except Exception as e:
        print(f"Failed: {e}")
        return None

pref_df = load_preference_dataset(PREF_DATASET)
if pref_df is None:
    PREF_DATASET_ALT = f"{USER_ID}/gpt2-imdb-seed{SEED}-prop{PROPORTION}-alpha{ALPHA}"
    pref_df = load_preference_dataset(PREF_DATASET_ALT)
    if pref_df is not None:
        PREF_DATASET = PREF_DATASET_ALT

if pref_df is not None:
    print(f"Preference dataset: {PREF_DATASET}")
    print(f"Shape: {pref_df.shape}")
    print(f"Columns: {pref_df.columns.tolist()}")

Preference dataset: mxcui/gpt2-imdb-preference-ALPHA1.0-len512-random
Shape: (24720, 3)
Columns: ['group_id', 'chosen', 'rejected']


In [8]:
# Compute length (word count as proxy) for chosen vs rejected
if pref_df is not None and 'chosen' in pref_df.columns and 'rejected' in pref_df.columns:
    pref_df['chosen_len'] = pref_df['chosen'].str.split().str.len()
    pref_df['rejected_len'] = pref_df['rejected'].str.split().str.len()
    
    print("Chosen vs Rejected length (word count):")
    print(pref_df[['chosen_len','rejected_len']].describe())
    print(f"\nChosen shorter than rejected: {(pref_df['chosen_len'] < pref_df['rejected_len']).mean()*100:.1f}%")
    print(f"Chosen longer than rejected:  {(pref_df['chosen_len'] > pref_df['rejected_len']).mean()*100:.1f}%")

Chosen vs Rejected length (word count):
         chosen_len  rejected_len
count  24720.000000  24720.000000
mean     103.290210    123.520469
std       96.306418    103.093126
min        1.000000      1.000000
25%       37.000000     45.000000
50%       69.000000     92.000000
75%      139.000000    171.000000
max      489.000000    489.000000

Chosen shorter than rejected: 58.6%
Chosen longer than rejected:  40.2%


## 3. Analysis: Why Does the Policy Learn Conciseness?

### Hypothesis 1: Short + Positive = "Pareto optimal" for the RM

The preference dataset uses **group-aware** scoring:
- **Minority (20%)**: score = (1-norm_len)*α + (1-α)*positiveness → prefers **conciseness**
- **Majority (80%)**: score = (1-norm_len)*(1-α) + α*positiveness → prefers **positiveness**

When α=1.0: minority fully prefers conciseness, majority fully prefers positiveness.

The vanilla RM trains on ALL pairs (no group info) and learns "chosen > rejected". Responses that are **both short AND positive** satisfy BOTH groups' preferences. E.g. "Great movie!" beats both long rambling text and short negative text. The RM may learn that short + positive = high reward. **Generating EOS early** is an easy way to produce short responses; if those short responses tend to be positive (e.g. "Amazing!", "Loved it!"), the policy gets high reward.

In [ ]:
# Check: among shortest eval responses, what's the positiveness distribution?
if 'positiveness' in df.columns:
    short_cutoff = df['token_length'].quantile(0.1)  # bottom 10%
    short_df = df[df['token_length'] <= short_cutoff]
    rest_df = df[df['token_length'] > short_cutoff]
    print("Positiveness: Short (bottom 10% by length) vs Rest")
    print(short_df['positiveness'].describe())
    print(rest_df['positiveness'].describe())
    print(f"\nMean positiveness - Short: {short_df['positiveness'].mean():.4f}, Rest: {rest_df['positiveness'].mean():.4f}")
    print("\nIf short responses have HIGH positiveness, this supports H1/H2: policy found short+positive = high reward.")

Positiveness: Short (bottom 10% by length) vs Rest
count    3551.000000
mean        0.953120
std         0.168504
min         0.003041
25%         0.992275
50%         0.994481
75%         0.995354
max         0.996393
Name: positiveness, dtype: float64
count    21321.000000
mean         0.976985
std          0.109626
min          0.003629
25%          0.993637
50%          0.994874
75%          0.995517
max          0.996541
Name: positiveness, dtype: float64

Mean positiveness - Short: 0.9531, Rest: 0.9770

If short responses have HIGH positiveness, this supports: policy found short+positive = high reward.


### Hypothesis 3: Base model + EOS bias

The base model (lvwerra/gpt2-imdb) is fine-tuned on IMDB. Short exclamations ("Great!", "Loved it!") may have high probability under the base model. PPO may simply **amplify** this tendency: the RM rewards short positive responses, so the policy learns to generate EOS sooner.

In [10]:
# Summary: length distribution in eval vs training
print("SUMMARY")
print("="*60)
print(f"Vanilla eval - mean length: {df['token_length'].mean():.1f} tokens")
print(f"Vanilla eval - median length: {df['token_length'].median():.1f} tokens")
if pref_df is not None:
    # Preference chosen/rejected are in words; *1.3 ~ tokens
    print(f"Preference chosen - mean ~{pref_df['chosen_len'].mean()*1.3:.0f} tokens (approx)")
    print(f"Preference rejected - mean ~{pref_df['rejected_len'].mean()*1.3:.0f} tokens (approx)")
print("\nKey question: Does the RM give higher scores to shorter responses?")
print("If so, the policy has learned to exploit that.")

SUMMARY
Vanilla eval - mean length: 18.6 tokens
Vanilla eval - median length: 18.0 tokens
Preference chosen - mean ~134 tokens (approx)
Preference rejected - mean ~161 tokens (approx)

Key question: Does the RM give higher scores to shorter responses?
If so, the policy has learned to exploit that.
